In [10]:
from pathlib import Path
import sys

# Locate 02_Model so that `v3.src` can be imported from anywhere in the repository.
working_directory = Path.cwd().resolve()
model_root = next(
    parent / "02_Model"
    for parent in (working_directory, *working_directory.parents)
    if (parent / "02_Model" / "v3" / "src").is_dir()
)
if str(model_root) not in sys.path:
    sys.path.insert(0, str(model_root))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from v3.src import create_mines, params
from v3.src.setup_model import build_model
from v3.src.params import Scenario, RefiningRoute

from v3.src import units

world_production_2024_t_li = 222_000.0
manufacturing_efficiency = 0.95  # Matches setup.py.

# Model demand uses final-product proxy tonnes.
final_demand_t = world_production_2024_t_li * units.LCE_PER_LI

config = {
    "initialisation_seed": 2,
    "simulation_seed": 1,
    "end_year": 2050,
    "operating_capacity_multiplier": 1.3,
    "demand_scenario" : "STEPS",
    "refinery_capacity_t_per_year": (
        final_demand_t / manufacturing_efficiency
    ),
}

scenario = Scenario(
    steps_per_year=1,  # 1 = annual; 4 = quarterly, 12 = monthly, 52 = weekly
    refined_price_per_t=25_000.0,
    mine_markup_fraction=0.0,
    refinery_required_markup_fraction=0.0,
    hard_rock_route=RefiningRoute(
        lithium_recovery_fraction=0.90,
        processing_cost_per_t_output=3_000.0,
    ),
    brine_route=RefiningRoute(
        lithium_recovery_fraction=0.85,
        processing_cost_per_t_output=3_500.0,
    ),
)

In [ ]:
raw_data = create_mines.load_raw_mines()

prepared_mines = create_mines.create_mines_dataset(
    raw_data,
    seed=config["initialisation_seed"],
    base_year=params.BASE_YEAR,
    historical_utilisation=params.HISTORICAL_UTILISATION,
)

from v3.src.iea_demand import load_iea_demand

iea = load_iea_demand()

demand_t_li = (
    iea.loc[iea["scenario"].eq(config["demand_scenario"])]
    .set_index("year")["demand_t_li"]
    .copy()
)

# Observed-production benchmark for the model's starting year.
demand_t_li.loc[2024] = world_production_2024_t_li

years = range(params.BASE_YEAR, config["end_year"] + 1)
demand_t_li = demand_t_li.reindex(years)

if demand_t_li.isna().any():
    raise ValueError("Demand is unavailable for some requested years.")

# Convert contained lithium to the existing final-product proxy units.
demand = {
    int(year): float(quantity * units.LCE_PER_LI)
    for year, quantity in demand_t_li.items()
}
# Always create fresh assets when changing the configuration.
model = build_model(
    prepared_mines,
    demand=demand,
    refinery_capacity=max(demand.values()) / manufacturing_efficiency,
    simulation_seed=config["simulation_seed"],
    scenario=scenario, 
    start_year=params.BASE_YEAR
)
for mine in model.mines:
    if mine.status == "operation":
        mine.capacity *= config["operating_capacity_multiplier"]

results = model.run()

trades = pd.DataFrame(
    model.transactions,
    columns=pd.Index([
        "period",
        "material_form",
        "source_asset_id",
        "quantity_t_input",
        "output_t",
        "price_per_t_input",
        "transaction_value",
    ]),
)
# Upstream purchases have an associated refined-output equivalent.
upstream = trades.loc[trades["output_t"].notna()].copy()

results

In [12]:
route_labels = {
    "spodumene_concentrate": "Hard-rock concentrate",
    "concentrated_brine": "Concentrated brine",
}
route_colours = {
    "spodumene_concentrate": "#2166ac",
    "concentrated_brine": "#e69f00",
}

periods = results["period"]

tonnes_proxy_per_kt_li = 1000.0 * units.LCE_PER_LI

year_by_period = results.set_index("period")["year"]

# Accepted sellers receive a common price within each material.
# Leave periods without trade as missing prices.
prices = (
    upstream.groupby(["period", "material_form"])["price_per_t_input"]
    .first()
    .unstack("material_form")
    .reindex(index=results["period"], columns=list(route_labels))
    .set_axis(results["time"], axis=0)
)

# Express both routes on the same refined-output basis.
volumes = (
    upstream.groupby(["period", "material_form"])["output_t"]
    .sum()
    .unstack("material_form")
    .reindex(index=results["period"], columns=list(route_labels))
    .fillna(0.0)
    .set_axis(results["time"], axis=0)
)

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Final demand and deliveries",
        "Upstream transaction prices",
        "Purchased supply by route",
        "Accumulated end-use stock",
    ),
    vertical_spacing=0.18,
)

for column, label, colour in [
    ("demand_t", "Demand", "#555555"),
    ("delivered_t", "Delivered", "#228833"),
]:
    fig.add_trace(
        go.Scatter(
            x=results["time"],
            y=results[column] / tonnes_proxy_per_kt_li,
            name=label,
            mode="lines+markers",
            line={"color": colour},
        ),
        row=1, col=1,
    )

for material, label in route_labels.items():
    fig.add_trace(
        go.Scatter(
            x=results["time"],
            y=prices[material],
            name=label,
            legendgroup=material,
            mode="lines+markers",
            connectgaps=False,
            line={"color": route_colours[material]},
        ),
        row=1, col=2,
    )

    fig.add_trace(
        go.Bar(
            x=results["time"],
            y=volumes[material] / tonnes_proxy_per_kt_li,
            name=label,
            legendgroup=material,
            showlegend=False,
            marker_color=route_colours[material],
        ),
        row=2, col=1,
    )

fig.add_trace(
    go.Scatter(
        x=results["time"],
        y=results["in_use_t"] / tonnes_proxy_per_kt_li,
        name="In use",
        mode="lines+markers",
        line={"color": "#884488"},
    ),
    row=2, col=2,
)

fig.update_yaxes(
    title_text="kt Li", row=1, col=1
)
fig.update_yaxes(
    title_text="USD/t physical input", row=1, col=2
)
fig.update_yaxes(
    title_text="kt Li, refined-output equivalent", row=2, col=1
)
fig.update_yaxes(
    title_text="kt Li", row=2, col=2
)
fig.update_xaxes(title_text="Year", dtick=5)
fig.update_yaxes(rangemode="tozero")

fig.update_layout(
    title="Supply-chain experiment",
    template="plotly_white",
    height=750,
    barmode="stack",
    hovermode="x unified",
)

fig.show()